In [11]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
from openai import OpenAI
import gradio as gr
import json

In [2]:
groq_api_key=os.getenv('GROQ_API_KEY')
groq = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1")

In [3]:
system_message="You are a helpful assistant for airline called WrightAIR." \
"Give short answers, not more than one sentence." \
"Always be accurate .If you dont know the answer say no"

In [ ]:
def chat(message,history):
    history=[{"role":h["role"],"content":h["content"]} for h in history]
    messages=[{"role":"system","content":system_message}]+history+[{"role":"user","content":message}]
    response=groq.chat.completions.create(model="openai/gpt-oss-120b",messages=messages)
    return response.choices[0].message.content

In [5]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


tools


In [7]:
ticket_prices = {
    "London": "$1200",
    "Tokyo": "$1500",
    "Paris": "$1300",
    "New York": "$1400",
    "Dubai": "$1000",
    "Singapore": "$1100",
    "Sydney": "$1600",
    "Toronto": "$1250",
    "Berlin": "$1150",
    "Rome": "$1050"
}
def get_ticket_price(destination_city):
    print(f"Tool called for {destination_city}")
    price=ticket_prices.get(destination_city,"price unknown")
    return f"The price of the ticket to {destination_city} is {price}"
    

In [8]:
price_function={
    "name":"get_ticket_price",
    "description":"get the price of a return ticket to the destination city.",
    "parameters":{
        "type":"object",
        "properties":{
            "destination_city":{
                "type":"string",
                "description":"The city that the customer wants to travel to",
            },
        },
        "required":["destination_city"],
        "additionalProperties":False
    }
}

In [9]:
tools=[{"type":"function","function":price_function}]

In [10]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [18]:
def chat(message,history):
    history=[{"role":h["role"],"content":h["content"]} for h in history]
    messages=[{"role":"system","content":system_message}]+history+[{"role":"user","content":message}]
    response=groq.chat.completions.create(model="openai/gpt-oss-120b",messages=messages,tools=tools,tool_choice="auto")

    if response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        responses=handle_tool_call(message)
        messages.append(message)
        messages.extend(responses)
        response=groq.chat.completions.create(model="openai/gpt-oss-120b",messages=messages)
    return response.choices[0].message.content


In [19]:
def handle_tool_call(message):
    responses=[]
    for tool_call in message.tool_calls:
        if tool_call.function.name=="get_ticket_price":
            arguments=json.loads(tool_call.function.arguments)
            city=arguments.get('destination_city')
            price_details=get_ticket_price(city)
            responses.append({"role":"tool","content":price_details,"tool_call_id":tool_call.id})
    return responses

In [21]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [22]:
import sqlite3

In [39]:
conn = sqlite3.connect("prices.db")
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS prices (
    city TEXT PRIMARY KEY,
    price TEXT
)
""")
conn.commit()
conn.close()

In [40]:
def get_ticket_price(city):
    print(f"database tool called : getting price for {city}")
    conn = sqlite3.connect("prices.db")
    cursor = conn.cursor()
    cursor.execute(
        "SELECT price FROM prices WHERE city=?",
        (city.lower(),)
    )
    row = cursor.fetchone()
    conn.close()
    if row:
        return f"The price of the ticket to {city} is {row[0]}"
    return "Price unknown"

In [41]:
def set_ticket_price(city, price):
    conn = sqlite3.connect("prices.db")
    cursor = conn.cursor()

    cursor.execute(
        """
        INSERT INTO prices(city, price)
        VALUES (?, ?)
        ON CONFLICT(city)
        DO UPDATE SET price=excluded.price
        """,
        (city.lower(), price)
    )

    conn.commit()
    conn.close()

In [42]:
for city,price in ticket_prices.items():
    set_ticket_price(city,price)

In [43]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


database tool called : getting price for London
